# LLaMA-2

[llama2](pics/llama2.png)

---

## 一、Rotary Positional Encoding

假设$d$维嵌入向量为$x$, token位置为$m$, 旋转矩阵为$\mathbf{R}_{\theta, m}^d$, 则经过旋转位置编码的向量$\mathbf{R}_{\theta, m}^d x$为：
$$
\mathbf{R}_{\theta, m}^d x = 
\begin{bmatrix}
x_1 \\ x_2 \\ x_3 \\ x_4 \\ \dots \\ x_{d-1} \\ x_d
\end{bmatrix} \otimes 
\begin{bmatrix}
\cos(m\theta_1) \\ \cos(m\theta_1) \\ \cos(m\theta_2) \\ \cos(m\theta_2) \\ \dots \\ \cos(m\theta_{d/2}) \\ \cos(m\theta_{d/2})
\end{bmatrix} +
\begin{bmatrix}
-x_2 \\ x_1 \\ -x_4 \\ x_3 \\ \dots \\ -x_{d} \\ x_{d-1}
\end{bmatrix} \otimes
\begin{bmatrix}
\sin(m\theta_1) \\ \sin(m\theta_1) \\ \sin(m\theta_2) \\ \sin(m\theta_2) \\ \dots \\ \sin(m\theta_{d/2}) \\ \sin(m\theta_{d/2})
\end{bmatrix}
$$
其中$\theta_i = 10000^{-2(i-1)/d}$, $d$维数被划分为相邻两两一组。这种计算方法**比直接进行旋转矩阵乘法快得多**。

In [1]:
import torch
import torch.nn as nn

"""
旋转位置编码: 一种相对位置编码，通过对原始向量进行旋转操作得到位置编码
对于一个query向量Wq*x, 通过对其乘以一个**旋转矩阵R**得到对应的旋转位置编码 
"""
def precompute_theta_pos_frequencies(dim: int, seq_len: int, device: str):
    assert dim % 2 == 0, "dimension must be even"
    
    """
    theta.shape: (dim/2)
    """
    i_iter = torch.arange(0, dim, 2).float()
    theta = 1.0 / (10000 ** (i_iter / dim)).to(device)
    
    """
    m.shape(position): (seq_len)
    freqs: (seq_len) @ (dim/2) -> (seq_len, dim/2), 即每个位置m都和所有的theta组合相乘, 得到正余弦内的数值
    """
    m = torch.arange(seq_len).to(device)
    freqs = torch.outer(m, theta).float()
    
    """
    torch.polar: 构建一个复数张量, 其元素模长均为1, 其元素角度来自freqs
    freqs_complex.shape: (seq_len, dim/2); 一个角度为mθ, 则对应的元素为cos(mθ)+i*sin(mθ)
    """
    freqs_complex = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_complex

那么如何将一个输入嵌入向量$x$按公式(1)转换为带有位置编码信息的旋转向量呢？假设$d = 4$, 且只有一个位置$m$。

1. 借助复数向量, 将输入$x$的维度**相邻两两组合**, 维度变成dim/2 (与freq的维度保持一致)：
$$
\begin{bmatrix}
x_1 \\ x_2 \\ x_3 \\ x_4
\end{bmatrix} \Rightarrow \begin{bmatrix}
x_1 + i x_2 \\ x_3 + i x_4
\end{bmatrix}
$$
- 另外由上面的freq复数化得到的复数张量为:
$$
\begin{bmatrix}
m\theta_1 \\ m\theta_2
\end{bmatrix} \Rightarrow
\begin{bmatrix}
\cos(m\theta_1) + i \sin(m\theta_1) \\
\cos(m\theta_2) + i \sin(m\theta_2) \\
\end{bmatrix}
$$

2. 将上述两个张量进行**逐元素相乘**, 得到旋转后的复数矩阵:
$$
\begin{bmatrix}
x_1 \cos(m\theta_1) - x_2 \sin(m\theta_1) + i [x_2 \cos(m\theta_1) + x_1\sin(m\theta_1)] \\
x_3 \cos(m\theta_2) - x_4 \sin(m\theta_2) + i [x_4 \cos(m\theta_2) + x_3\sin(m\theta_2)] \\
\end{bmatrix}
$$

3. 复向量**实数化**, 也就是拉直:
$$
\begin{bmatrix}
x_1 \cos(m\theta_1) - x_2 \sin(m\theta_1) \\
x_2 \cos(m\theta_1) + x_1 \sin(m\theta_1) \\
x_3 \cos(m\theta_2) - x_4 \sin(m\theta_2) \\
x_4 \cos(m\theta_2) + x_3 \sin(m\theta_2) \\
\end{bmatrix}
$$
- 可见上述公式**与开头旋转编码定义的公式计算结果完全相同**, 具体代码实现如下。

In [2]:
def apply_rotatary_embeddings(x: torch.Tensor, freqs_complex: torch.Tensor, device: str):
    """
    先将x的最后一维扩成2个 (对应一组实数+复数)
        (batch, seq_len, n_heads, dim) -> (batch, seq_len, n_heads, dim/2, 2)
    再将x转化为复数, 与freqs_complex形状相同
        (batch, seq_len, n_heads, dim/2, 2) -> (batch, seq_len, n_heads, dim/2)
    """
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    
    """
    先扩展出批次和多头的维度, 便于与x_complex逐元素相乘
        (seq_len, dim/2) -> (1, seq_len, 1, dim/2)
    再与x_complex逐元素相乘
        (1, seq_len, 1, dim/2) * (batch, seq_len, n_heads, dim/2) -> (batch, seq_len, n_heads, dim/2)
    """
    freqs_complex = freqs_complex.unsqueeze(0).unsqueeze(2)
    x_rotated = x_complex * freqs_complex
    
    """
    先添加末尾2维度, 还原回实数张量
        (batch, seq_len, n_heads, dim/2) -> (batch, seq_len, n_heads, dim/2, 2)
    再拉直, 还原成输入x的形状
        (batch, seq_len, n_heads, dim/2, 2) -> (batch, seq_len, n_heads, dim)
    """
    x_out = torch.view_as_real(x_rotated)
    x_out = x_out.reshape(*x.shape)
    
    return x_out.type_as(x).to(device)

---

## 二、RMS Normalization

RMSNorm是一种更加高效的样本归一化方法, 相较于层归一化方法其具有如下优势: 
1. 只需要计算一个统计量, 无需计算均值和方差
2. 不居中值, 而是缩小值, 效果更优

$$
\bar{a}_i = \dfrac{a_i}{\textbf{RMS}(a)} g_i, \qquad \text{where} \quad \textbf{RMS}(a) = \sqrt{\dfrac{1}{N} \sum_{i=1}^{N} a_i^2}
$$
其中 $g_i$ 是一个**可学习参数**。

In [3]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float=1e-6):
        super().__init__()
        self.eps = eps  # 防止除以0
        self.g = nn.Parameter(torch.ones(dim))
    
    def _norm(self, x: torch.Tensor):
        """
        rescale: (batch, seq_len, dim) / (batch, seq_len, 1) -> (batch, seq_len, dim)
        """
        return x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x: torch.Tensor):
        """
        (dim) * (batch, seq_len, dim) -> (batch, seq_len, dim)
        """
        return self.g * self._norm(x.float()).type_as(x)

---

## 三、Self-Attention

In [ ]:
from dataclasses import dataclass
from typing import Optional

"""
@dataclass: python装饰器, 定义一个类用来存储数据
自动生成__init__、__repr__等方法, 减少代码编写量
"""
@dataclass
class ModelArgs:
    dim: int = 4096
    num_layers: int = 32
    num_q_heads: int = 32
    num_kv_heads: Optional[int] = None  # q, kv的在GQA中的多头数量可以不一样
    vocab_size: int = 30000
    multiple_of: int = 256
    ffn_dim_multiplier: Optional[float] = None
    norm_eps: float = 1e-5
    max_batch_size: int = 32
    max_seq_len: int = 2048
    device: str = None

In [ ]:
import math
import torch.nn.functional as F


class SelfAttention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.num_kv_heads = args.num_q_heads if args.num_kv_heads is None else args.num_kv_heads
        self.num_q_heads = args.num_q_heads
        self.num_group = args.num_q_heads // args.num_kv_heads
        self.head_dim = args.dim // args.num_q_heads
        
        self.wq = nn.Linear(args.dim, args.num_q_heads * self.head_dim, bias=False, device=args.device)
        self.wk = nn.Linear(args.dim, args.num_kv_heads * self.head_dim, bias=False, device=args.device)
        self.wv = nn.Linear(args.dim, args.num_kv_heads * self.head_dim, bias=False, device=args.device)
        self.wo = nn.Linear(args.num_q_heads * self.head_dim, args.dim, bias=False, device=args.device)
        
        self.cache_k = torch.zeros(size=(args.max_batch_size, args.max_seq_len, args.num_kv_heads, self.head_dim), device=args.device)
        self.cache_v = torch.zeros(size=(args.max_batch_size, args.max_seq_len, args.num_kv_heads, self.head_dim), device=args.device)
        
    def repeat_kv(self, x: torch.Tensor, num_group: int):
        if num_group == 1:
            return x
        else:
            batch_size, seq_len, num_kv_heads, head_dim = x.shape
            """
            先引入一个新维度1: (batch, 1, kv_heads, 1, head_dim)
            再将kv头复制group份, 从而与q对齐: (batch, 1, kv_heads, num_group, head_dim) -> (batch, 1, kv_heads * num_group, head_dim)
            """
            x = x[:, :, :, None, :].expand(batch_size, seq_len, num_kv_heads, num_group, head_dim)
            x = x.reshape(batch_size, seq_len, num_kv_heads * num_group, head_dim)
            return x
        
    def forward(
        self, 
        x: torch.Tensor, 
        start_pos: int, # 推理时指定当前的生成位置, 训练时始终为0
        freqs_complex: torch.Tensor,
        mask: Optional[torch.Tensor] = None
    ):
        batch_size, seq_len, _ = x.shape # (batch, seq_len, dim)
        
        """
        q:  (batch, seq_len, dim) -> (batch, seq_len, q_heads * head_dim)
        kv: (batch, seq_len, dim) -> (batch, seq_len, kv_heads * head_dim)
        """
        xq = self.wq(x)
        xk = self.wk(x)
        xv = self.wv(x)
        
        """
        q:  (batch, seq_len, q_heads * head_dim) -> (batch, seq_len, q_heads, head_dim)
        kv: (batch, seq_len, kv_heads * head_dim) -> (batch, seq_len, kv_heads, head_dim)
        """
        xq = xq.view(batch_size, seq_len, self.num_q_heads, self.head_dim)
        xk = xk.view(batch_size, seq_len, self.num_kv_heads, self.head_dim)
        xv = xv.view(batch_size, seq_len, self.num_kv_heads, self.head_dim)
        
        """
        只对q和k进行旋转编码, 不改变张量形状:
        q: (batch, seq_len, q_heads, head_dim)
        k: (batch, seq_len, kv_heads, head_dim)
        """
        xq = apply_rotatary_embeddings(x=xq, freqs_complex=freqs_complex, device=x.device)
        xk = apply_rotatary_embeddings(x=xk, freqs_complex=freqs_complex, device=x.device)
        
        """
        先将当前输入的kv序列(seq_len)压入kv cache中; 推理阶段seq_len=1
        """
        self.cache_k[:batch_size, start_pos:start_pos+seq_len] = xk
        self.cache_v[:batch_size, start_pos:start_pos+seq_len] = xv
        
        """
        再将cache中的所有缓存token取出, kv_cache: (batch, cache_len+seq_len, kv_heads, head_dim)
        """
        keys = self.cache_k[:batch_size, 0:start_pos+seq_len]
        values = self.cache_v[:batch_size, 0:start_pos+seq_len]
        
        """
        将kv均复制group份, 从而与q的多头数量q_heads对齐:
            kv: (batch, cache_len+seq_len, kv_heads, head_dim) -> (batch, cache_len+seq_len, q_heads, head_dim)
        """
        keys = self.repeat_kv(keys, self.num_group)
        values = self.repeat_kv(values, self.num_group)
        
        """
        将q, k, v分别转置, 从而让每个head都能看到所有的序列seq(的一部分)
        q:  (batch, seq_len, q_heads, head_dim) -> (batch, q_heads, seq_len, head_dim)
        kv: (batch, cache_len+seq_len, q_heads, head_dim) -> (batch, q_heads, cache_len+seq_len, head_dim)
        可以看出q的序列长度仅为seq_len(推理时=1); 但kv的序列长度为cache_len+seq_len(推理时=cache_len+1), 即通过kv cache缓存了先前的序列
        """
        xq = xq.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)
        
        """
        计算自注意力分数
        (batch, q_heads, seq_len, head_dim) @ (batch, q_heads, head_dim, cache_len+seq_len)
            -> (batch, q_heads, seq_len, cache_len+seq_len)
        """
        scores = torch.matmul(xq, keys.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores + mask  # (batch, q_heads, seq_len, cache_len+seq_len)
        scores = F.softmax(scores.float(), dim=-1).type_as(xq)
        
        """
        先将注意力score乘以多头values:
        (batch, q_heads, seq_len, cache_len+seq_len) @ (batch, q_heads, cache_len+seq_len, head_dim)
            -> (batch, q_heads, seq_len, head_dim)
        """
        output = torch.matmul(scores, values)
        """
        再将多头输出output拼接起来:
            (batch, q_heads, seq_len, head_dim) -> (batch, seq_len, q_heads, head_dim) -拼接-> (batch, seq_len, dim)
        最后乘以Wo矩阵, 输出形状保持不变: 
            (batch, seq_len, dim) -> (batch, seq_len, dim)
        推理时seq_len=1, 即输出的只有下一个token
        """
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        return self.wo(output)

---

## 四、SwiGLU

SwiGLU是Swish和GLU的合并版本, 其中GLU是一种门控层, 由两层线性层$W$和$V$组成:
$$
\text{GLU}(x, W, V, b, c) = \sigma(Wx+b) \otimes (Vx+c)
$$
将其激活函数从$\sigma$改为$\text{Swish}_1$, 可以得到SwiGLU激活函数。继续叠加一个线性层, 可以得到FFN层。

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        """
        将前馈层隐层参数量扩展4倍, 再乘以2/3(SwiGLU有三个线性层), 即8/3倍
        并将hidden_dim**向上对齐**到multiple_of的最近整数倍
        """
        hidden_dim = 8 * args.dim // 3
        if args.ffn_dim_multiplier is not None:
            hidden_dim = int(args.ffn_dim_multiplier * hidden_dim)
        hidden_dim = args.multiple_of * ((hidden_dim + args.multiple_of - 1) // args.multiple_of)
        
        self.w1 = nn.Linear(args.dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, args.dim, bias=False)
        self.w3 = nn.Linear(args.dim, hidden_dim, bias=False)
        
    def forward(self, x: torch.Tensor):
        """
        (batch, seq_len, dim) -> (batch, seq_len, hidden_dim) -> (batch, seq_len, dim)
        推理时seq_len=1, 即输出的只有下一个token
        """
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

---

## 五、Decoder Block 和 Decoder

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.num_heads = args.num_q_heads
        self.dim = args.dim
        self.dim_head = args.dim // args.num_q_heads
        
        self.attention = SelfAttention(args)
        self.ffn = FeedForward(args)
        
        self.attention_norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.ffn_norm = RMSNorm(args.dim, eps=args.norm_eps)
        
    def forward(self, x: torch.Tensor, start_pos: int, freqs_complex: torch.Tensor, mask: Optional[torch.Tensor] = None):
        """
        (batch_size, seq_len, dim) + (batch_size, seq_len, dim) -> (batch_size, seq_len, dim)
        推理时seq_len=1, 即输出的只有下一个token
        """
        h = x + self.attention(self.attention_norm(x), start_pos, freqs_complex, mask)
        out = h + self.ffn(self.ffn_norm(h))
        return out
    
    
class Decoder(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.vocab_size = args.vocab_size
        self.num_layers = args.num_layers
        self.token_embedding = nn.Embedding(args.vocab_size, args.dim)
        
        self.layers = nn.ModuleList()
        for _ in range(args.num_layers):
            self.layers.append(DecoderBlock(args))
            
        self.norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.proj = nn.Linear(args.dim, args.vocab_size, bias=False)
        self.freqs_complex = precompute_theta_pos_frequencies(
                                self.args.dim // self.args.num_q_heads, 
                                self.args.max_seq_len*2, # 预留2倍的上下文
                                device=self.args.device
                            )
        
    def forward(self, tokens: torch.Tensor, start_pos: int, targets=None):
        batch_size, seq_len = tokens.shape
        
        """
        转化为词嵌入向量: (batch_size, seq_len) -> (batch_size, seq_len, dim)
        """
        h = self.token_embedding(tokens)
        freq_complex = self.freqs_complex[start_pos : start_pos+seq_len]
        
        """
        如果不提供targets, 则为推理模式(inference), 无需causal_mask
        否则causal_mask用于训练模式(train), 其形状为(seq_len, cache_len+seq_len), 也即mask能控制模型仅可见缓存的prompt+当前输入
        """
        if targets is None:
            mask = None
        else:
            mask = torch.full(size=(seq_len, seq_len), full_value=float("-inf"), device=tokens.device)
            mask = torch.triu(mask, diagonal=1, device=tokens.device).type_as(h)
            
        for layer in self.layers:
            h = layer(h, start_pos, freq_complex, mask)
        h = self.norm(h)
        
        """
        将模型输出分数转化为词表上的分数: (batch, seq_len, dim) -> (batch, seq_len, vocab_size)
        推理阶段seq_len=1, 训练阶段seq_len>1
        """
        logits = self.proj(h).float()
        
        if targets is None:
            loss = None
        else:
            loss = F.cross_entropy(
                logits.view(-1, logits.shape[-1]), 
                targets.view(-1),
            )
        
        return logits, loss

---

## 六、Inference

In [ ]:
from sentencepiece import SentencePieceProcessor

tokenizer = SentencePieceProcessor()
tokenizer.Load('[PATH TO tokenizer.model]')
args = ModelArgs(
    max_seq_len=32,
    num_q_heads=32,
    num_kv_heads=16,
    vocab_size=tokenizer.vocab_size(), 
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
model = Decoder(args).cuda()


def sample_top_p(probs, top_p):  # Top-P策略, 又称核采样策略
    """
    对候选词的probs进行降序排序, 进而从高到低计算累计和
        probs_sort/idx: (batch, vocab_size) -> (batch, vocab_size)
    排序后, probs_idx即为vocab中的索引值, prob_sort对应了prob_idx的概率值
    """
    probs_sort, probs_idx = torch.sort(probs, dim=-1, descending=True)
    probs_sum = torch.cumsum(probs_sort, dim=-1)
    
    """
    筛选出累计概率大于等于top_p的token的最小集合, 并将剩余的小概率置0
    """
    mask = probs_sum > top_p
    probs_sort[mask] = 0.0
    
    """
    对筛选出的新概率分布进行重新归一化, 并基于概率从每行采样一个token
        (batch, vocab_size) -> (batch, 1), 采得的其实是行内索引
    gather函数沿最后一维从probs_idx中提取[行内索引next_token_idx]指定的元素, 即最终的token_id
    """
    probs_sort.div_(probs_sort.sum(dim=-1, keepdim=True))
    next_token_idx = torch.multinomial(probs_sort, num_samples=1)
    next_token = torch.gather(probs_idx, dim=-1, index=next_token_idx)
    return next_token
    

def text_completion(
    prompts: list[str],
    temperature: float = 0.0,
    top_p: float = 0.8,
    max_seq_len: Optional[int] = None,
):
    if max_seq_len is None:
        max_seq_len = args.max_seq_len - 1
    
    prompt_tokens = [tokenizer.Encode(prompt, out_type=int, add_bos=True, add_eos=False) for prompt in prompts]
    batch_size = len(prompt_tokens)
    max_prompt_len = max(len(prompt) for prompt in prompt_tokens)
    assert batch_size <= args.max_batch_size, "batch size exceeds max batch size"
    assert max_prompt_len <= args.max_seq_len, "max prompt length exceeds max seq length"
    
    """
    tokens: 存储推理得到的tokens, 用初始的提示词初始化
    eos_reached: 用于标记每个推理样本是否已经到达了结束符
    prompt_tokens_mask: pad掩码, 标记每个样本的输入提示词部分
    """
    total_len = min(args.max_seq_len, max_seq_len + max_prompt_len)
    pad_id = tokenizer.pad_id()
    tokens = torch.full(size=(batch_size, total_len), fill_value=pad_id, dtype=torch.long, device=args.device)
    for k, t in enumerate(prompt_tokens):
        tokens[k, :len(t)] = torch.tensor(t, dtype=torch.long, device=args.device)
        
    eos_reached = torch.tensor([False] * batch_size, device=args.device)
    prompt_tokens_mask = tokens != pad_id
    
    for cur_pos in range(1, total_len):
        with torch.no_grad():
            """
            一次只传入一个已有的token(其位置为cur_pos-1), 得到next_token的logits
                (batch, seq_len=1) -> (batch, seq_len=1, vocab_size)
            同时传入cur_pos, 用来更新kv_cache
            """ 
            logits, _ = model(tokens[:, cur_pos-1:cur_pos], start_pos=cur_pos, targets=None)
        if temperature > 0:
            """
            如果指定了temperature, 采用Top-P策略
                (batch, seq_len=1, vocab_size) -取最后一个token-> (batch, vocab_size) -取最大值索引-> (batch,)
            """
            probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
            next_token = sample_top_p(probs=probs, top_p=top_p).reshape(-1)
        else:
            """
            greedy decoding:
                (batch, seq_len=1, vocab_size) -取最后一个token-> (batch, vocab_size) -取最大值索引-> (batch,)
            """
            next_token = torch.argmax(logits[:, -1], dim=-1)
        
        """
        如果当前预测位置cur_pos的token不是pad, 则保留原输入prompt tokens不变
        如果当前预测位置cur_pos的token是pad, 说明已经超出了原始输入prompt的范围, 则将当前位置填为预测的next_token
        """
        next_token = torch.where(prompt_tokens_mask[:, cur_pos], tokens[:, cur_pos], next_token)
        tokens[:, cur_pos] = next_token
        
        """
        当前位置不是pad & 预测的下一token是eos -> 值为真, 同时更新eos_reached从False到True
        如果所有的eos_reached都为True, 说明批次内的所有prompt都已经生成完毕
        """
        eos_reached |= (~prompt_tokens_mask[:, cur_pos]) & (next_token == tokenizer.eos_id())
        if all(eos_reached):
            break
    
    out_tokens = []
    out_texts = []
    for _, cur_prompt_tokens in enumerate(tokens.tolist()):
        """
        如果生成的一句tokens中存在eos, 找到其位置并截断到eos
        """
        if tokenizer.eos_id() in cur_prompt_tokens:
            eos_idx = cur_prompt_tokens.index(tokenizer.eos_id())
            cur_prompt_tokens = cur_prompt_tokens[:eos_idx]
        out_tokens.append(cur_prompt_tokens)
        out_texts.append(tokenizer.Decode(cur_prompt_tokens))
    
    return out_texts

prompts = ["What is the meaning of life", "Introduce yourself in English."]
print(text_completion(prompts=prompts))

['What is the meaning of lifeailand IB Television Влади vomznik abandoned tr memor passe主 personasyanuidtabularтет találкла yourselfagan IBulaire gentleoverline prot', 'Introduce yourself in English. pitch gminie)\\generatorängen radial bet LCCN Eing Germ passesктивո indicate décadaüs� aveva Ga Dokument calculated арmetadata Tomatoes']
